# 🔄 Generate Synthetic Data for LLM-JEPA Symbolic Regression

This notebook generates large-scale synthetic physics equations for pretraining the LLM-JEPA model.

**What this does:**
- Clones/pulls the repository
- Syncs to Google Drive (SymbolicRegression folder)
- Generates synthetic equations with physics-informed constraints
- Saves to cache/synthetic_1M (or configured path)

**Runtime:** T4 GPU recommended, but CPU works too

---

In [ ]:
# @title 📦 Setup: Clone Repo & Sync to Google Drive

import os
import subprocess
from pathlib import Path
from google.colab import drive

# Mount Google Drive
print("📁 Mounting Google Drive...")
drive.mount('/content/drive')

# Define paths
DRIVE_FOLDER = "/content/drive/MyDrive/SymbolicRegression"
WORK_DIR = "/content/GSOC-LM-JEPA_for_Symbolic_Regression"
REPO_URL = "https://github.com/udohchuks/GSOC-LM-JEPA_for_Symbolic_Regression.git"

# Create Drive folder if not exists
Path(DRIVE_FOLDER).mkdir(parents=True, exist_ok=True)
print(f"✅ Drive folder ready: {DRIVE_FOLDER}")

# Clone or pull repository
if os.path.exists(WORK_DIR):
    print("📦 Repository found, pulling latest changes...")
    os.chdir(WORK_DIR)
    subprocess.run(["git", "pull"], check=True)
else:
    print("📦 Cloning repository...")
    os.chdir("/content")
    subprocess.run(["git", "clone", REPO_URL], check=True)
    os.chdir(WORK_DIR)

# Sync to Drive (copy working directory)
print("🔄 Syncing to Google Drive...")
sync_target = f"{DRIVE_FOLDER}/code"
Path(sync_target).mkdir(parents=True, exist_ok=True)
subprocess.run(["rsync", "-av", "--delete", f"{WORK_DIR}/", f"{sync_target}/"], check=True)
print(f"✅ Synced to: {sync_target}")

# Install dependencies
print("📦 Installing dependencies...")
os.chdir(WORK_DIR)
subprocess.run(["pip", "install", "-q", "-r", "requirements.txt"], check=True)
print("✅ Dependencies installed")

# Create cache directory in Drive
cache_dir = f"{DRIVE_FOLDER}/cache"
Path(cache_dir).mkdir(parents=True, exist_ok=True)
print(f"✅ Cache directory ready: {cache_dir}")

In [ ]:
# @title ⚙️ Configuration

# @markdown ### Generation Parameters
N_SYNTHETIC = 20000  # @param {type: "integer"}
N_DATA_POINTS = 500  # @param {type: "integer"}
NUM_WORKERS = 4  # @param {type: "integer"}
CHUNK_SIZE = 100  # @param {type: "integer"}

# @markdown ### Presets
# @markdown - **20k equations** (for ~1M model): N_SYNTHETIC=20000, N_DATA_POINTS=500, CHUNK_SIZE=100
# @markdown - **100k equations** (for 3.4M model): N_SYNTHETIC=100000, N_DATA_POINTS=2000, CHUNK_SIZE=500
# @markdown - **1M equations** (full scale): N_SYNTHETIC=1000000, N_DATA_POINTS=2000, CHUNK_SIZE=1000

# @markdown ### Output Location
CACHE_DIR = f"{DRIVE_FOLDER}/cache"  # @param {type: "string"}
SYNTHETIC_SUBFOLDER = "synthetic_20k"  # @param {type: "string"}

# Update config file
import yaml

config_path = f"{WORK_DIR}/configs/small.yaml"  # Use small config
with open(config_path, 'r') as f:
    config = yaml.safe_load(f)

# Update data generation params
config['data']['n_synthetic'] = N_SYNTHETIC
config['data']['n_data_points'] = N_DATA_POINTS
config['data']['num_workers'] = NUM_WORKERS
config['data']['chunk_size'] = CHUNK_SIZE
config['data']['synthetic_cache'] = f"{CACHE_DIR}/{SYNTHETIC_SUBFOLDER}"

# Save updated config
with open(config_path, 'w') as f:
    yaml.dump(config, f)

print(f"✅ Configuration updated:")
print(f"   - Equations: {N_SYNTHETIC:,}")
print(f"   - Data points per equation: {N_DATA_POINTS:,}")
print(f"   - Chunk size: {CHUNK_SIZE}")
print(f"   - Cache location: {CACHE_DIR}/{SYNTHETIC_SUBFOLDER}")

In [ ]:
# @title 🚀 Generate Synthetic Data

import time
import yaml
from pathlib import Path

print(f"🔄 Starting synthetic data generation...")
print(f"   Target: {N_SYNTHETIC:,} equations")
print(f"   Data points per equation: {N_DATA_POINTS}")
print(f"   Workers: {NUM_WORKERS}")
print(f"   Chunk size: {CHUNK_SIZE} equations/file")
print(f"   Output: {CACHE_DIR}/{SYNTHETIC_SUBFOLDER}")
print(f"   Config: configs/small.yaml")
print("=" * 60)
print()
print("📊 Live generation progress will appear below...")
print("   Look for: 'Generating X/100 equations' and yield %")
print()

start_time = time.time()

# Update config with Drive path
config_path = f"{WORK_DIR}/configs/small.yaml"
with open(config_path, 'r') as f:
    config = yaml.safe_load(f)

# Override cache path to use Drive
config['data']['synthetic_cache'] = f"{CACHE_DIR}/{SYNTHETIC_SUBFOLDER}"
config['data']['n_synthetic'] = N_SYNTHETIC
config['data']['n_data_points'] = N_DATA_POINTS
config['data']['num_workers'] = NUM_WORKERS
config['data']['chunk_size'] = CHUNK_SIZE

# Save updated config
with open(config_path, 'w') as f:
    yaml.dump(config, f)

print(f"✅ Config updated:")
print(f"   Cache path: {config['data']['synthetic_cache']}")
print(f"   N_synthetic: {N_SYNTHETIC:,}")
print(f"   Chunk size: {CHUNK_SIZE}")
print()

# Run generation with verbose output
%cd $WORK_DIR
!python -m data.generate_data --config configs/small.yaml

elapsed = time.time() - start_time
hours = elapsed / 3600

print("\n" + "=" * 60)
if len([l for l in result if 'Error' in l or 'Traceback' in l]) > 0:
    print("❌ Generation FAILED - see errors above")
    print()
    print("🔧 Common causes:")
    print("   1. Config file not found: Check configs/small.yaml exists")
    print("   2. Out of memory: Reduce N_DATA_POINTS or NUM_WORKERS")
    print("   3. Disk full: Check Drive storage")
else:
    print(f"✅ Generation complete!")
    print(f"   Time elapsed: {hours:.2f} hours ({elapsed:.0f} seconds)")
    
    # Count generated files
    output_path = Path(f"{CACHE_DIR}/{SYNTHETIC_SUBFOLDER}")
    if output_path.exists():
        n_files = len(list(output_path.glob("*.pt")))
        total_size = sum(f.stat().st_size for f in output_path.glob("*.pt"))
        total_eq = n_files * CHUNK_SIZE
        print()
        print(f"📈 Generation Summary:")
        print(f"   Files generated: {n_files}")
        print(f"   Total equations: ~{total_eq:,}")
        print(f"   Total size: {total_size / (1024**2):.1f} MB ({total_size / (1024**3):.2f} GB)")
        print(f"   Equations per file: {CHUNK_SIZE}")
        print(f"   Yield rate: Check 'done: X/Y' in output above")
    else:
        print("\n⚠️ Output directory not found. Check for errors above.")

In [ ]:
# @title 📊 Verify Generated Data (Optional)

import torch
from pathlib import Path

cache_path = Path(f"{CACHE_DIR}/{SYNTHETIC_SUBFOLDER}")

if not cache_path.exists():
    print("❌ Cache directory not found. Run generation first.")
else:
    # Find all .pt files
    pt_files = list(cache_path.glob("*.pt"))
    print(f"📁 Found {len(pt_files)} data files")
    
    # Load and inspect first file
    if pt_files:
        print(f"\n📄 Inspecting: {pt_files[0].name}")
        data = torch.load(str(pt_files[0]), weights_only=False)
        
        if isinstance(data, list):
            print(f"   Type: List of {len(data)} equations")
            if len(data) > 0:
                eq = data[0]
                print(f"   Sample equation:")
                print(f"      - Variables: {eq.n_vars if hasattr(eq, 'n_vars') else 'N/A'}")
                print(f"      - X_bits shape: {eq.X_bits.shape if hasattr(eq, 'X_bits') else 'N/A'}")
                print(f"      - Expression: {eq.expr_str if hasattr(eq, 'expr_str') else 'N/A'}")
        else:
            print(f"   Type: {type(data)}")
            print(f"   Shape/Keys: {data.shape if hasattr(data, 'shape') else data.keys() if isinstance(data, dict) else 'N/A'}")
    
    # Total size
    total_size = sum(f.stat().st_size for f in pt_files)
    print(f"\n💾 Total cache size: {total_size / (1024**3):.2f} GB")

---
## Next Steps

After generating synthetic data:

1. **Training**: Use [`02_train_model.ipynb`](02_train_model.ipynb) to pretrain on this data
2. **Evaluation**: Use [`03_evaluate_model.ipynb`](03_evaluate_model.ipynb) to test on AI Feynman benchmark

## Tips

- **Large-scale generation (1M+)**: Takes ~4-8 hours on Colab T4
- **Keep runtime alive**: Use Colab's keep-alive feature or connect to local runtime
- **Data saved to Drive**: All generated data persists in `SymbolicRegression/cache/`